# 第311章: ノイズ空間補間

## 📋 この章で学ぶこと

- [ ] DDIM InversionとSamplingを組み合わせた画像変容パイプラインを実装できる
- [ ] ノイズ空間での線形補間とSlerp補間を比較し、その違いを説明できる
- [ ] 異なるタイムステップでの補間が変容の「粗さ」に与える影響を理解できる
- [ ] 拡散モデルによる画像変容のGANとの違いを定量的に比較できる

## 🎯 前提知識

- ✅ Notebook 310（DDIM Inversion）
- ✅ Notebook 306（Slerp補間）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 実践

---

## 🌟 はじめに

310章ではDDIM Inversionで「画像→ノイズ→再構成」を学びました。
この章では、2つの画像のInversionノイズを**補間**することで、
高品質な画像変容を実現します。

### パイプライン全体像

```
  画像A ──→ DDIM Inversion ──→ xₐ_T（ノイズA）
                                    │
                                    ▼ 補間 (α)
  画像B ──→ DDIM Inversion ──→ x_B_T（ノイズB）
                                    │
                                    ▼
                             x_T(α) = Lerp or Slerp
                                    │
                                    ▼ DDIM Sampling
                             変容画像 x₀(α)
```

### 📝 この章の構成

1. **モデルと学習** — 310章と同じDDIM設定
2. **基本パイプライン** — Inversion + 補間 + Sampling
3. **Lerp vs Slerp** — ノイズ空間での補間手法の比較
4. **タイムステップ選択の影響** — t=T vs t=T/2
5. **品質評価** — SSIM推移、中間画像の品質分析

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
print(f"Device: {device}")
print("✅ 環境設定完了")

In [ ]:
# ============================================================
# 拡散モデルの学習（310章と同一設定）
# ============================================================

def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return {
        'betas': torch.tensor(betas, dtype=torch.float32),
        'alphas': torch.tensor(alphas, dtype=torch.float32),
        'alpha_bars': torch.tensor(alpha_bars, dtype=torch.float32),
    }

class NoisePredictor(nn.Module):
    def __init__(self, dim=784, time_emb_dim=32):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_emb_dim), nn.SiLU(), nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(dim + time_emb_dim, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, dim),
        )
    def forward(self, x, t):
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        return self.net(torch.cat([x, t_emb], dim=-1))

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

schedule = get_schedule(n_steps=100)
noise_pred = NoisePredictor().to(device)
optimizer = optim.Adam(noise_pred.parameters(), lr=2e-4)
alpha_bars = schedule['alpha_bars'].to(device)

print("拡散モデル学習中...")
for epoch in range(2):
    noise_pred.train()
    total = 0
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device) * 2 - 1
        t = torch.randint(0, 100, (x.shape[0],)).to(device)
        ab = alpha_bars[t].unsqueeze(-1)
        noise = torch.randn_like(x)
        x_noisy = torch.sqrt(ab) * x + torch.sqrt(1 - ab) * noise
        loss = nn.functional.mse_loss(noise_pred(x_noisy, t), noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15 | Loss: {total/len(train_loader):.6f}")
print("✅ 学習完了")

In [ ]:
# ============================================================
# DDIM Inversion と Sampling の関数定義
# ============================================================

@torch.no_grad()
def ddim_inversion(model, x_0, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    x = x_0.clone()
    for i in range(n_steps):
        t_c, t_n = step_idx[i], step_idx[i + 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model(x, t_t)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_n]) * x0_p + torch.sqrt(1 - ab[t_n]) * eps
    return x

@torch.no_grad()
def ddim_sample(model, x_T, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    x = x_T.clone()
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model(x, t_t)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

def slerp(z1, z2, alpha):
    z1_n = z1 / (z1.norm(dim=-1, keepdim=True) + 1e-10)
    z2_n = z2 / (z2.norm(dim=-1, keepdim=True) + 1e-10)
    dot = (z1_n * z2_n).sum(dim=-1, keepdim=True).clamp(-1, 1)
    omega = torch.acos(dot)
    s = torch.sin(omega)
    # fallback to lerp where s is very small
    use_lerp = (s < 1e-10)
    c1 = torch.where(use_lerp, torch.ones_like(omega) * (1 - alpha),
                     torch.sin((1 - alpha) * omega) / (s + 1e-10))
    c2 = torch.where(use_lerp, torch.ones_like(omega) * alpha,
                     torch.sin(alpha * omega) / (s + 1e-10))
    norm = (1 - alpha) * z1.norm(dim=-1, keepdim=True) + alpha * z2.norm(dim=-1, keepdim=True)
    direction = c1 * z1_n + c2 * z2_n
    direction = direction / (direction.norm(dim=-1, keepdim=True) + 1e-10)
    return direction * norm

noise_pred.eval()
# テスト画像の準備
test_batch, test_labels = next(iter(test_loader))
test_imgs = test_batch.view(-1, 784).to(device) * 2 - 1

print("✅ 補間関数定義完了")

---

## 1. 基本パイプライン: Lerp補間

In [ ]:
# ============================================================
# Inversion + Lerp補間 + Sampling
# ============================================================

# 5ペアの数字を選択
pairs = [(test_labels == i).nonzero()[0].item() for i in range(5)]
pair_imgs_a = test_imgs[pairs]

# B側：次の数字
pairs_b = [(test_labels == (i+1)%10).nonzero()[0].item() for i in range(5)]
pair_imgs_b = test_imgs[pairs_b]

# Inversion
print("Inversion中...")
noise_a = ddim_inversion(noise_pred, pair_imgs_a, n_steps=5)
noise_b = ddim_inversion(noise_pred, pair_imgs_b, n_steps=5)

# Lerp補間
n_interp = 9
alphas = torch.linspace(0, 1, n_interp)

fig, axes = plt.subplots(5, n_interp, figsize=(16, 9))

print("Sampling中...")
for row in range(5):
    na = noise_a[row:row+1]
    nb = noise_b[row:row+1]
    for col, alpha in enumerate(alphas):
        a = alpha.item()
        z_interp = (1 - a) * na + a * nb   # Lerp
        x_gen = ddim_sample(noise_pred, z_interp, n_steps=5)
        img = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0: axes[row, col].set_title(f'α={a:.2f}', fontsize=9)
    label_a = test_labels[pairs[row]].item()
    label_b = test_labels[pairs_b[row]].item()
    axes[row, 0].set_ylabel(f'{label_a}→{label_b}', fontsize=11, rotation=0, labelpad=35)

fig.suptitle('DDIM Inversion + ノイズ空間Lerp補間
左(α=0): 元画像A  右(α=1): 元画像B',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_311_01_lerp_morphing.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Lerp変容完了")

---

## 2. Lerp vs Slerp 補間の比較

In [ ]:
# ============================================================
# Lerp vs Slerp の比較
# ============================================================

fig, axes = plt.subplots(2, n_interp, figsize=(16, 5))
row_idx = 0
na_s = noise_a[row_idx:row_idx+1]
nb_s = noise_b[row_idx:row_idx+1]

label_a = test_labels[pairs[row_idx]].item()
label_b = test_labels[pairs_b[row_idx]].item()

for col, alpha in enumerate(alphas):
    a = alpha.item()
    # Lerp
    z_l = (1 - a) * na_s + a * nb_s
    img_l = ((ddim_sample(noise_pred, z_l, n_steps=5)[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[0, col].imshow(img_l, cmap='gray')
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel('Lerp', fontsize=13, rotation=0, labelpad=35)
    axes[0, col].set_title(f'{a:.2f}', fontsize=9)

    # Slerp
    z_s = slerp(na_s, nb_s, a)
    img_s = ((ddim_sample(noise_pred, z_s, n_steps=5)[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[1, col].imshow(img_s, cmap='gray')
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel('Slerp', fontsize=13, rotation=0, labelpad=35)

fig.suptitle(f'ノイズ空間 Lerp vs Slerp ({label_a}→{label_b})',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_311_02_lerp_vs_slerp.png', dpi=150, bbox_inches='tight')
plt.show()

# ノルム変化の確認
lerp_norms = [(((1-a)*na_s + a*nb_s).norm(dim=-1).mean().item()) for a in alphas.numpy()]
slerp_norms = [(slerp(na_s, nb_s, a.item()).norm(dim=-1).mean().item()) for a in alphas]

fig2, ax2 = plt.subplots(1,1, figsize=(10,5))
ax2.plot(alphas.numpy(), lerp_norms, 'b-o', linewidth=2, label='Lerp')
ax2.plot(alphas.numpy(), slerp_norms, 'r-s', linewidth=2, label='Slerp')
ax2.set_xlabel('α', fontsize=13); ax2.set_ylabel('ノイズノルム', fontsize=13)
ax2.set_title('補間中のノイズノルム変化', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_311_03_noise_norms.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Lerp中点ノルム: {lerp_norms[4]:.2f}  Slerp中点ノルム: {slerp_norms[4]:.2f}")

---

## 3. タイムステップ選択の影響

Inversionを途中（t=T×0.5）で止めると、変容の「粗さ」が変わります。

```
  t=T（フルInversion）: 全情報をノイズ化 → 大きな変化
  t=T/2（半分）:        低周波成分のみノイズ化 → 細部を保持した変容
```

In [ ]:
# ============================================================
# タイムステップ数の影響
# ============================================================

fig, axes = plt.subplots(4, n_interp, figsize=(16, 8))

n_steps_list = [10, 25, 50, 100]
row_idx = 0
img_a_s = pair_imgs_a[row_idx:row_idx+1]
img_b_s = pair_imgs_b[row_idx:row_idx+1]

for r, n_inv in enumerate(n_steps_list):
    na_t = ddim_inversion(noise_pred, img_a_s, n_steps=n_inv)
    nb_t = ddim_inversion(noise_pred, img_b_s, n_steps=n_inv)

    for col, alpha in enumerate(alphas):
        a = alpha.item()
        z_interp = slerp(na_t, nb_t, a)
        x_gen = ddim_sample(noise_pred, z_interp, n_steps=n_inv)
        img = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
        axes[r, col].imshow(img, cmap='gray')
        axes[r, col].axis('off')
        if col == 0: axes[r, col].set_ylabel(f'{n_inv}ステップ', fontsize=10, rotation=0, labelpad=45)
        if r == 0: axes[r, col].set_title(f'α={a:.2f}', fontsize=8)

fig.suptitle('Inversionステップ数が変容品質に与える影響
多いステップ = 高精度だが計算コスト大',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_311_04_timestep_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 ステップ数が多いほど元画像の再構成精度が高く、より自然な変容を実現")

In [ ]:
# ============================================================
# SSIM推移の計算
# ============================================================

def ssim_simple(a, b):
    c1, c2 = 0.01**2, 0.03**2
    mu_a, mu_b = a.mean(), b.mean()
    va, vb = a.var(), b.var()
    cov = ((a - mu_a)*(b - mu_b)).mean()
    return ((2*mu_a*mu_b+c1)*(2*cov+c2)) / ((mu_a**2+mu_b**2+c1)*(va+vb+c2))

img_a_orig = ((pair_imgs_a[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
img_b_orig = ((pair_imgs_b[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)

na_f = noise_a[0:1]; nb_f = noise_b[0:1]
ssim_lerp, ssim_slerp = [], []
alphas_eval = torch.linspace(0, 1, 15)

for alpha in alphas_eval:
    a = alpha.item()
    # Lerp
    zl = (1-a)*na_f + a*nb_f
    il = ((ddim_sample(noise_pred, zl, n_steps=5)[0].cpu().numpy().reshape(28, 28)+1)/2).clip(0,1)
    ssim_lerp.append(ssim_simple(il, img_a_orig) if a < 0.5 else ssim_simple(il, img_b_orig))
    # Slerp
    zs = slerp(na_f, nb_f, a)
    is_ = ((ddim_sample(noise_pred, zs, n_steps=5)[0].cpu().numpy().reshape(28, 28)+1)/2).clip(0,1)
    ssim_slerp.append(ssim_simple(is_, img_a_orig) if a < 0.5 else ssim_simple(is_, img_b_orig))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(alphas_eval.numpy(), ssim_lerp, 'b-o', linewidth=2, label='Lerp SSIM')
ax.plot(alphas_eval.numpy(), ssim_slerp, 'r-s', linewidth=2, label='Slerp SSIM')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='中点 α=0.5')
ax.set_xlabel('α', fontsize=13); ax.set_ylabel('SSIM（近い端との類似度）', fontsize=12)
ax.set_title('補間過程のSSIM推移', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_311_05_ssim_transition.png', dpi=150, bbox_inches='tight')
plt.show()

---

## まとめ

### 🎯 学んだこと

- ✓ DDIM Inversion + ノイズ補間 + DDIM Sampling の3段パイプライン
- ✓ ノイズ空間でもSlerp補間が有効（高次元正規分布のため）
- ✓ Inversionステップ数が多いほど変容品質が向上するが計算コスト増

### ⚠️ よくある間違い

❌ 「ノイズ空間Lerpで十分」
✅ ノイズベクトルも高次元正規分布からくるため、Slerpのほうがノルムが安定する

### ✅ 学習チェックリスト

- [ ] Inversion+補間+Samplingのパイプラインを説明できるか？
- [ ] タイムステップ数が変容品質に与える影響を説明できるか？

---

## 🎓 自己評価クイズ

### Q1: 拡散モデルのノイズ空間での補間と、VAEの潜在空間での補間の本質的な違いは？

<details>
<summary>💡 答えを見る</summary>

**答え**: 拡散モデルのノイズ空間はエンコーダで直接得られる。VAEの潜在空間はKL正則化された低次元表現。拡散モデルのノイズはほぼ全情報を保持し、高精度な再構成が可能。

</details>

---

### Q2: Inversionのステップ数を増やすとなぜ再構成品質が向上するか？

<details>
<summary>💡 答えを見る</summary>

**答え**: ステップ数が多いほど離散化誤差が小さくなり、DDIMの連続時間方程式への近似精度が上がるため。ステップ数が少ないと、各ステップでの誤差が累積して再構成品質が低下する。

</details>